# transformer architecture 

In [22]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-31 10:03:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-07-31 10:03:00 (125 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [23]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [24]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [25]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [26]:
# ---------- Byte Pair Encoding (BPE) Tokenizer ----------
# Inspired by Andrej Karpathy's minbpe.
# Instead of the character-level tokenizer above, BPE starts
# with raw bytes (0-255) and *learns* merges from the data,
# producing a compact, subword vocabulary.

# ---- helpers ----

def get_stats(ids):
    """Count every consecutive pair in `ids`.
       e.g. [1,2,3,1,2] → {(1,2):2, (2,3):1, (3,1):1}"""
    counts = {}
    for pair in zip(ids, ids[1:]):       # sliding window of 2
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`.
       e.g. ids=[1,2,3,1,2], pair=(1,2), new_id=4 → [4,3,4]"""
    out = []
    i = 0
    while i < len(ids):
        # if we're not at the last element AND the pair matches → merge
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            out.append(new_id)
            i += 2                       # skip both tokens
        else:
            out.append(ids[i])
            i += 1
    return out


# ---- tokenizer class ----

class BPETokenizer:
    """Minimal byte-level BPE tokenizer."""

    def __init__(self):
        self.merges = {}      # (int, int) → int   learned merge rules
        self.vocab  = {}      # int → bytes         id → token bytes

    # -- training --

    def train(self, text, vocab_size, verbose=False):
        """Learn `vocab_size - 256` merges from `text`."""
        assert vocab_size >= 256, "vocab_size must be ≥ 256"
        num_merges = vocab_size - 256

        # step 1: encode text to raw UTF-8 bytes
        tokens = list(text.encode("utf-8"))

        # step 2: iteratively merge the most frequent pair
        merges = {}
        vocab  = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            stats = get_stats(tokens)
            if not stats:
                break
            best_pair = max(stats, key=stats.get)     # highest-freq pair
            new_id    = 256 + i                       # new token id
            tokens    = merge(tokens, best_pair, new_id)
            merges[best_pair] = new_id
            vocab[new_id]     = vocab[best_pair[0]] + vocab[best_pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {best_pair} → {new_id}  "
                      f"({vocab[new_id]})  (appeared {stats[best_pair]}×)")

        self.merges = merges
        self.vocab  = vocab

    # -- encoding --

    def encode(self, text):
        """String → list[int]  (apply learned merges)."""
        tokens = list(text.encode("utf-8"))
        while len(tokens) >= 2:
            stats = get_stats(tokens)
            # pick the pair that was merged *earliest* during training
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break                    # nothing left to merge
            tokens = merge(tokens, pair, self.merges[pair])
        return tokens

    # -- decoding --

    def decode(self, ids):
        """list[int] → string."""
        raw = b"".join(self.vocab[i] for i in ids)
        return raw.decode("utf-8", errors="replace")


# ---------- demo ----------

# train on the Shakespeare text loaded above
vocab_size = 300                # 256 byte tokens + 44 learned merges
tokenizer  = BPETokenizer()
tokenizer.train(text, vocab_size, verbose=True)

# encode / decode round-trip
sample = "hii there"
ids    = tokenizer.encode(sample)
back   = tokenizer.decode(ids)

print("\n--- round-trip test ---")
print(f"original : {sample!r}")
print(f"token ids: {ids}")
print(f"decoded  : {back!r}")
print(f"match    : {sample == back}")

# compression ratio  (raw bytes vs token count)
raw_len = len(sample.encode("utf-8"))
print(f"bytes {raw_len} → tokens {len(ids)}  "
      f"(compression {raw_len / len(ids):.2f}×)")

merge 1/44: (101, 32) → 256  (b'e ')  (appeared 27643×)
merge 2/44: (116, 104) → 257  (b'th')  (appeared 22739×)
merge 3/44: (116, 32) → 258  (b't ')  (appeared 16508×)
merge 4/44: (115, 32) → 259  (b's ')  (appeared 15364×)
merge 5/44: (100, 32) → 260  (b'd ')  (appeared 14165×)
merge 6/44: (44, 32) → 261  (b', ')  (appeared 14098×)
merge 7/44: (111, 117) → 262  (b'ou')  (appeared 12730×)
merge 8/44: (101, 114) → 263  (b'er')  (appeared 11771×)
merge 9/44: (105, 110) → 264  (b'in')  (appeared 10606×)
merge 10/44: (121, 32) → 265  (b'y ')  (appeared 10283×)
merge 11/44: (97, 110) → 266  (b'an')  (appeared 10197×)
merge 12/44: (58, 10) → 267  (b':\n')  (appeared 8762×)
merge 13/44: (111, 114) → 268  (b'or')  (appeared 8458×)
merge 14/44: (111, 32) → 269  (b'o ')  (appeared 8134×)
merge 15/44: (101, 110) → 270  (b'en')  (appeared 7568×)
merge 16/44: (10, 10) → 271  (b'\n\n')  (appeared 7098×)
merge 17/44: (97, 114) → 272  (b'ar')  (appeared 7081×)
merge 18/44: (32, 257) → 273  (b' th')  

In [27]:
import torch 
print(torch.__version__)

2.11.0+cu128


In [28]:
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([788667]) torch.int64


In [29]:
print(data[:100])

tensor([ 70, 299, 115, 258,  67, 105, 116, 105, 122, 270, 267,  66, 101, 102,
        268, 256, 119, 256, 112, 114, 111,  99, 101, 101, 260, 266, 265, 102,
        117, 114, 257, 263, 261, 104, 101, 272, 287, 256, 115, 112, 286, 107,
        278,  65, 275, 267,  83, 112, 286, 107, 261, 115, 112, 286, 107, 278,
         70, 299, 115, 258,  67, 105, 116, 105, 122, 270, 267,  89, 262,  32,
        272, 256,  97, 275,  32, 114, 280, 111, 108, 118, 101, 260, 114,  97,
        257, 263,  32, 283, 100, 105, 256, 257, 266,  32, 283, 102,  97, 109,
        105, 115])


In [30]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [31]:
print(len(train_data),len(val_data))

709800 78867


In [32]:
block_size = 8 
train_data[:block_size+1]

tensor([ 70, 299, 115, 258,  67, 105, 116, 105, 122])

In [33]:
# ---------- Building Batches for a Transformer ----------
# We need to feed the model fixed-length chunks of token IDs.
# For a language model, the TARGET is just the INPUT shifted by 1.
#
# block_size = 8  (already defined above — the max context length)
batch_size = 4   # how many independent sequences per batch

def get_batch(split):
    """
    Sample a random batch of (input, target) pairs from the data.

    Returns:
        x: (batch_size, block_size)   — input  token IDs
        y: (batch_size, block_size)   — target token IDs  (x shifted right by 1)

    ── dry run (batch_size=4, block_size=8) ──

    Suppose train_data = [70, 299, 115, 258, 67, 105, 116, 105, 122, 270, 267, 66, ...]
                          (709800 tokens total)

    Step 1 — pick random starting positions:
        torch.randint(high=709800 - 8, size=(4,))
        say we get:  ix = tensor([41203, 5884, 600321, 12005])

    Step 2 — for each start index, slice a window of block_size tokens for x,
             and a window shifted by 1 for y:

        i = 41203:
            x[0] = train_data[41203 : 41203+8]  →  8 tokens
            y[0] = train_data[41204 : 41204+8]  →  8 tokens (same window, shifted right by 1)

        i = 5884:
            x[1] = train_data[5884 : 5892]
            y[1] = train_data[5885 : 5893]

        ... same for indices 600321 and 12005 ...

    Step 3 — stack into tensors:
        x.shape = (4, 8)   — 4 sequences, each 8 tokens long
        y.shape = (4, 8)

    Why shift by 1?
        At every position t in x, the model should predict y[t]:
            x = [70, 299, 115, 258,  67, 105, 116, 105]
            y = [299, 115, 258,  67, 105, 116, 105, 122]
                 ↑                                        
        given [70]           → predict 299
        given [70, 299]      → predict 115
        given [70, 299, 115] → predict 258
        ...and so on up to the full context window.

    This single chunk actually gives us block_size=8 training examples
    (each with a different context length from 1 to block_size).
    """
    src = train_data if split == 'train' else val_data

    # random start indices — never go past the end
    #   high = len(src) - block_size  so that src[i : i+block_size] is in bounds
    ix = torch.randint(len(src) - block_size, (batch_size,))

    # build x and y by stacking slices
    # build x and y by collecting slices into lists, then stacking
    x_slices = []
    y_slices = []
    for i in ix:
        x_slices.append(src[i     : i + block_size])
        y_slices.append(src[i + 1 : i + block_size + 1])
    x = torch.stack(x_slices)
    y = torch.stack(y_slices)
    x, y = x.to(device), y.to(device)
    return x, y

# ---------- quick sanity check ----------
xb, yb = get_batch('train')
print("inputs  shape :", xb.shape)   # (4, 8)
print("targets shape :", yb.shape)   # (4, 8)

print("\n--- full batch ---")
print("xb =\n", xb)
print("yb =\n", yb)

# --- walk through every training example inside this batch ---
# Each row gives block_size examples (context → target):
print("\n--- unpacking the training signal from row 0 ---")
for t in range(block_size):
    context = xb[0, :t+1]     # first 1, then first 2, ... up to all 8
    target  = yb[0, t]        # the single next token to predict
    print(f"  context {str(context.tolist()):>40s}  →  target {target.item()}")

inputs  shape : torch.Size([4, 8])
targets shape : torch.Size([4, 8])

--- full batch ---
xb =
 tensor([[ 59, 273, 256, 115, 270,  97, 116, 101],
        [288, 105, 257, 273, 101, 265, 119, 105],
        [ 69,  82,  77,  73,  79,  78,  69, 267],
        [108, 260, 112, 117, 258, 281,  32, 283]], device='cuda:0')
yb =
 tensor([[273, 256, 115, 270,  97, 116, 101,  10],
        [105, 257, 273, 101, 265, 119, 105, 275],
        [ 82,  77,  73,  79,  78,  69, 267,  87],
        [260, 112, 117, 258, 281,  32, 283, 281]], device='cuda:0')

--- unpacking the training signal from row 0 ---
  context                                     [59]  →  target 273
  context                                [59, 273]  →  target 256
  context                           [59, 273, 256]  →  target 115
  context                      [59, 273, 256, 115]  →  target 270
  context                 [59, 273, 256, 115, 270]  →  target 97
  context             [59, 273, 256, 115, 270, 97]  →  target 116
  context        

In [34]:
## karpathy way 
xx = train_data[:block_size]
yy = train_data[1:block_size+1]
for t in range(block_size):
    context = xx[:t+1]
    target = yy[t]
    print(f"input: {context} | target: {target}")



input: tensor([70]) | target: 299
input: tensor([ 70, 299]) | target: 115
input: tensor([ 70, 299, 115]) | target: 258
input: tensor([ 70, 299, 115, 258]) | target: 67
input: tensor([ 70, 299, 115, 258,  67]) | target: 105
input: tensor([ 70, 299, 115, 258,  67, 105]) | target: 116
input: tensor([ 70, 299, 115, 258,  67, 105, 116]) | target: 105
input: tensor([ 70, 299, 115, 258,  67, 105, 116, 105]) | target: 122


## Autoregressive Language Modeling — The Math

### 1. The Goal: Model a Sequence's Probability

Given a sequence of tokens $x_1, x_2, \ldots, x_T$, we want to model:

$$P(x_1, x_2, \ldots, x_T)$$

By the **chain rule of probability**, this factors as:

$$P(x_1, x_2, \ldots, x_T) = P(x_1) \cdot P(x_2 \mid x_1) \cdot P(x_3 \mid x_1, x_2) \cdots P(x_T \mid x_1, \ldots, x_{T-1}) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

Each factor $P(x_t \mid x_{<t})$ says: *"given all previous tokens, what's the probability of the next one?"*  
This is **exactly** what the shift-by-1 sets up as the training target.

---

### 2. How the Shift-by-1 Maps to This

```
x = [x₁,  x₂,  x₃,  x₄,  x₅,  x₆,  x₇,  x₈ ]   ← input
y = [x₂,  x₃,  x₄,  x₅,  x₆,  x₇,  x₈,  x₉ ]   ← target (shifted right by 1)
```

At each position $t$, the model outputs logits, then softmax gives a distribution over the vocab:

$$\hat{y}_t = \text{softmax}\big(f_\theta(x_1, \ldots, x_t)\big)$$

The target says: at position $t$, the correct answer is $y_t = x_{t+1}$.  
So the model learns: $P_\theta(x_{t+1} \mid x_1, \ldots, x_t)$ for every $t$ **simultaneously** in one forward pass.

---

### 3. The Loss: Cross-Entropy

At position $t$:

$$\mathcal{L}_t = -\log P_\theta(x_{t+1} \mid x_1, \ldots, x_t)$$

- Model assigns **high probability** to the right token → $-\log(\text{high}) \approx 0$ → low loss ✓  
- Model assigns **low probability** → $-\log(\text{low}) \gg 0$ → high loss ✗

Average over the full sequence:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P_\theta(x_{t+1} \mid x_1, \ldots, x_t)$$

> Minimizing this = maximizing the likelihood of the data under the model.

---

### 4. Why the Causal Mask is Necessary

Self-attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

The $QK^\top$ matrix is $(T \times T)$ — entry $(t, t')$ = how much position $t$ attends to $t'$.  
**Without a mask**, position $t=1$ can see $t'=2,3,4$ — it sees the future!

**Fix:** Add a causal mask $M$ before softmax:

$$M = \begin{bmatrix} 0 & -\infty & -\infty & -\infty \\ 0 & 0 & -\infty & -\infty \\ 0 & 0 & 0 & -\infty \\ 0 & 0 & 0 & 0 \end{bmatrix}$$

$$\text{Attention} = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

After $-\infty$ goes through softmax → those weights become **exactly 0**.  
Now position $t$ can only attend to $t' \leq t$ — it genuinely depends only on the past. ✓

---

### 5. Summary

```
x = [x₁, x₂, x₃, x₄] → Transformer (with causal mask) → logits → softmax → P(next | past)
                                                                                    │
                         compare with y = [x₂, x₃, x₄, x₅] via cross-entropy ◄────┘
                                                    │
                              loss = -1/T Σ log P(xₜ₊₁ | x₁...xₜ)
                                                    │
                              backprop → update θ → model gets better
```

| Component | Mathematical role |
|---|---|
| **Shift-by-1** | Defines targets $y_t = x_{t+1}$ |
| **Causal mask** | Enforces $P(x_t \mid x_{<t})$, not $P(x_t \mid x_{\neq t})$ |
| **Cross-entropy** | Computes $-\log P_\theta(x_{t+1} \mid x_{\leq t})$ |
| **Chain rule** | Product of all conditionals = full sequence probability |

> At **inference**, you only have past tokens. The causal mask during training ensures the model **never learned to rely on future tokens**, so generation works correctly.

In [35]:
# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

In [36]:
"""
coding the self attention
q --> what i am looking for 
k --> what i have 
so the dot product tells the affinity of this 

we use scaled self attention because it makes the variance unity and mean zero 
"""

import torch
import torch.nn as nn 
import torch.nn.functional as F 

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # this mean this quantity will not be used to have weights that will train 

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out


In [37]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        
        # --- WITHOUT LIST COMPREHENSION ---
        self.heads = nn.ModuleList()
        for _ in range(num_heads):
            self.heads.append(Head(head_size))
            
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # --- WITHOUT LIST COMPREHENSION ---
        head_outputs = []
        for h in self.heads:
            out_single_head = h(x)
            head_outputs.append(out_single_head)
            
        # Concatenate outputs from all heads along channel dimension
        out = torch.cat(head_outputs, dim=-1)
        
        out = self.dropout(self.proj(out))
        return out


In [38]:
class FeedFoward(nn.Module):
    """ 
    a simple linear layer followed by a non-linearity 
    it helps in thinking about the data individually 
    """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [39]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [40]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # Step 1: Create a regular list
        blocks_list = []

        # Step 2: Loop n_layer times and create 6 Block instances
        for _ in range(n_layer):
            new_block = Block(n_embd, n_head=n_head)
            blocks_list.append(new_block)

        # Step 3: Unpack the list into nn.Sequential using '*'
        self.blocks = nn.Sequential(*blocks_list)

        self.ln_f = nn.LayerNorm(n_embd) # final layer norm 
        # layer norm because ach token can be normalized independently, so the result does not change merely because different sentences were placed beside it in the batch
        self.lm_head = nn.Linear(n_embd, vocab_size)


        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [41]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [42]:
model = GPTLanguageModel()
m = model.to(device)

# Print parameter count
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# Create PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # Evaluate loss periodically
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample batch and send to device
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)  # <--- Moved to device

    # Forward pass & optimization
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_ids = m.generate(context, max_new_tokens=500)[0].tolist()

print(tokenizer.decode(generated_ids))  # <--- Used tokenizer.decode


10.969644 M parameters
step 0: train loss 5.7200, val loss 5.7038
step 500: train loss 2.1592, val loss 2.4603
step 1000: train loss 1.8123, val loss 2.2068
step 1500: train loss 1.6442, val loss 2.1542
step 2000: train loss 1.4989, val loss 2.1463


KeyboardInterrupt: 